# 03 — Drift detector + Lakehouse Monitoring

Two things happen here:

1. **Schema-drift detector view** — a SQL view that compares the keys present in `bronze.plays_raw.payload` (VARIANT) against the `known_payload_keys` allow-list. Any unknown keys come out one row each. This view is what the DBSQL Alert in `alerts/drift_alert.sql` watches.
2. **Lakehouse Monitor on `silver.plays`** — a snapshot-style monitor that profiles the typed silver columns. Once it's created, the dashboard appears in Catalog Explorer under the silver table → **Quality** tab.

These two cover complementary gaps:

| Pattern | Catches |
|---|---|
| Drift-detector view + DBSQL Alert | Brand-new top-level keys appearing in the API response (silver doesn't have these columns yet) |
| Lakehouse Monitoring | Distribution drift, freshness, completeness on columns that **already exist** in silver |

In [0]:
%pip install dotenv

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os, time
from dotenv import load_dotenv
load_dotenv()

try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

UC_CATALOG        = os.getenv('UC_CATALOG', 'alexander_booth')
UC_SCHEMA         = os.getenv('UC_SCHEMA',  'hockey_schema_monitoring')
BRONZE_TABLE      = f'{UC_CATALOG}.{UC_SCHEMA}_bronze.plays_raw'
SILVER_TABLE      = f'{UC_CATALOG}.{UC_SCHEMA}_silver.plays'
KNOWN_KEYS_TBL    = f'{UC_CATALOG}.{UC_SCHEMA}_silver.known_payload_keys'
DRIFT_VIEW        = f'{UC_CATALOG}.{UC_SCHEMA}_silver.v_unknown_payload_keys'
MONITORING_SCHEMA = os.getenv('MONITORING_SCHEMA') or f'{UC_SCHEMA}_silver_monitoring'

print(f'Drift view  : {DRIFT_VIEW}')
print(f'Silver      : {SILVER_TABLE}')
print(f'Monitoring  : {UC_CATALOG}.{MONITORING_SCHEMA}')

Drift view  : alexander_booth.hockey_schema_monitoring_silver.v_unknown_payload_keys
Silver      : alexander_booth.hockey_schema_monitoring_silver.plays
Monitoring  : alexander_booth.hockey_schema_monitoring_silver_monitoring


## Drift-detector view

Uses `variant_explode` to walk the top-level keys of every `payload`, then anti-joins the allow-list. Each row in the view = one unknown key sighting, with first/last-seen timestamps and a count so the alert can be tuned (e.g. fire only when seen > 10 times in the last hour).

In [0]:
spark.sql(f'''
    CREATE OR REPLACE VIEW {DRIFT_VIEW} AS
    WITH exploded AS (
        SELECT
            ingest_ts,
            source_version,
            kv.key AS payload_key
        FROM {BRONZE_TABLE},
        LATERAL variant_explode(payload) AS kv
    ),
    unknown AS (
        SELECT e.*
        FROM exploded e
        LEFT ANTI JOIN {KNOWN_KEYS_TBL} k ON k.key = e.payload_key
    )
    SELECT
        payload_key,
        COUNT(*)              AS sightings,
        MIN(ingest_ts)        AS first_seen,
        MAX(ingest_ts)        AS last_seen,
        COUNT(DISTINCT source_version) AS source_versions_affected,
        collect_set(source_version) AS source_versions
    FROM unknown
    GROUP BY payload_key
    ORDER BY last_seen DESC
''')

print('Drift view created. Current state (should be empty after v1 only):')
spark.sql(f'SELECT * FROM {DRIFT_VIEW}').show(truncate=False)

Drift view created. Current state (should be empty after v1 only):
+-----------+---------+----------+---------+------------------------+---------------+
|payload_key|sightings|first_seen|last_seen|source_versions_affected|source_versions|
+-----------+---------+----------+---------+------------------------+---------------+
+-----------+---------+----------+---------+------------------------+---------------+



## DBSQL Alert query

The alert query lives in `alerts/drift_alert.sql`. The Alerts UI will run it on a schedule; when the row count comes back > 0, it fires.

Print the query here so you can copy it during the demo without leaving the notebook.

In [0]:
alert_sql = f'''-- DBSQL Alert: unknown keys in bronze.plays_raw.payload
-- Fires when this query returns > 0 rows
SELECT
    payload_key,
    sightings,
    first_seen,
    last_seen
FROM {DRIFT_VIEW}
WHERE last_seen >= current_timestamp() - INTERVAL 1 HOUR
ORDER BY last_seen DESC;
'''
print(alert_sql)

-- DBSQL Alert: unknown keys in bronze.plays_raw.payload
-- Fires when this query returns > 0 rows
SELECT
    payload_key,
    sightings,
    first_seen,
    last_seen
FROM alexander_booth.hockey_schema_monitoring_silver.v_unknown_payload_keys
WHERE last_seen >= current_timestamp() - INTERVAL 1 HOUR
ORDER BY last_seen DESC;



## Lakehouse Monitoring on `silver.plays`

Snapshot-profile monitor. Databricks will:

1. Profile every column (null %, distinct count, distribution stats)
2. Compare each refresh to a baseline (here: the current state)
3. Render the dashboard at **Catalog Explorer → silver.plays → Quality tab**

The drift metrics table can itself feed a DBSQL Alert (e.g. "alert if `event_type` distribution JS-divergence > 0.2 vs baseline").

In [0]:
from databricks.sdk.service.catalog import MonitorInfoStatus

try:
    existing = w.lakehouse_monitors.get(SILVER_TABLE)
    print(f'Monitor already exists for {SILVER_TABLE} (status={existing.status})')
    print(f'Dashboard: {existing.dashboard_id}')
    print(f'Profile metrics table: {existing.profile_metrics_table_name}')
    print(f'Drift metrics table  : {existing.drift_metrics_table_name}')
except Exception:
    print(f'Creating monitor on {SILVER_TABLE} ...')
    monitor = w.lakehouse_monitors.create(
        full_name               = SILVER_TABLE,
        assets_dir              = f'/Workspace/Users/{w.current_user.me().user_name}/lhm/{UC_SCHEMA}',
        output_schema_name      = f'{UC_CATALOG}.{MONITORING_SCHEMA}',
        snapshot                = {},
    )
    print(f'Created. Dashboard ID: {monitor.dashboard_id}')
    print(f'Profile metrics: {monitor.profile_metrics_table_name}')
    print(f'Drift metrics  : {monitor.drift_metrics_table_name}')

Creating monitor on alexander_booth.hockey_schema_monitoring_silver.plays ...
Created. Dashboard ID: None
Profile metrics: alexander_booth.hockey_schema_monitoring_silver_monitoring.plays_profile_metrics
Drift metrics  : alexander_booth.hockey_schema_monitoring_silver_monitoring.plays_drift_metrics


In [0]:
# Kick off the first refresh so the dashboard has data when the user clicks into it.
# Newly-created monitors sit in PENDING for ~30–60s before they accept refreshes — wait for ACTIVE.
from databricks.sdk.service.catalog import MonitorInfoStatus
for _ in range(30):
    m = w.lakehouse_monitors.get(SILVER_TABLE)
    if m.status == MonitorInfoStatus.MONITOR_STATUS_ACTIVE:
        break
    print(f'  monitor status={m.status} — waiting...')
    time.sleep(5)

refresh = w.lakehouse_monitors.run_refresh(SILVER_TABLE)
print(f'Refresh queued: id={refresh.refresh_id} state={refresh.state}')
print('(it usually finishes in 1–3 minutes — check the Quality tab in Catalog Explorer)')

Refresh queued: id=823651803936946 state=MonitorRefreshInfoState.PENDING
(it usually finishes in 1–3 minutes — check the Quality tab in Catalog Explorer)
